# Dataset of the molecules clustered using the dictionnary 
## Two variants: **A-keep-odereless** vs **A-drop-odereless**

Base transformation 
1. Drop the 8 same-named fine labels: `citrus, earthy, floral, fruity, green, spicy, sweet, woody`
2. Keep their meta columns unchanged (This way we have a  partial independence of the parent from the children)


Odorless strategy differs between variants:

| | molecules | fine labels | meta labels |
|---|---|---|---|
| **A-keep** | 4976 | 129 | 13 (odorless added as childless meta) |
| **A-drop** | 4777 | 129 | 12 (odorless molecules removed) |


In [ ]:
import pandas as pd

df = pd.read_csv("hmcn_dataset.csv")

fine_cols = [c for c in df.columns if c.startswith("fine_")]
meta_cols = [c for c in df.columns if c.startswith("meta_")]
feat_cols = [c for c in df.columns if c not in fine_cols + meta_cols + ["SMILES"]]

print(f"Loaded: {len(df)} molecules | {len(feat_cols)} features | {len(fine_cols)} fine | {len(meta_cols)} meta")

## Step 1 — Drop the 8 same-named fine labels
Identical for both variants. The 216 molecules that had only one of these labels
become meta-only and are retained in both datasets.

In [ ]:
meta_names = {c.replace("meta_", "") for c in meta_cols}
fine_names = {c.replace("fine_", "") for c in fine_cols}
overlap    = sorted(meta_names & fine_names)
drop_same  = ["fine_" + n for n in overlap]

df = df.drop(columns=drop_same)

fine_now = [c for c in df.columns if c.startswith("fine_")]
meta_only_count = (df[fine_now].sum(axis=1) == 0).sum()
print(f"Dropped {len(drop_same)} same-named fine labels: {overlap}")
print(f"Remaining fine labels : {len(fine_now)}")
print(f"Meta-only molecules   : {meta_only_count}  (kept in both variants)")

## Step 2 — Identify odorless molecules

In [ ]:
odorless_mask = df["fine_odorless"] == 1
other_sum     = df[[c for c in fine_now if c != "fine_odorless"]].sum(axis=1)
purely_odorless = ((odorless_mask) & (other_sum == 0)).sum()
mixed           = ((odorless_mask) & (other_sum > 0)).sum()

print(f"Odorless total         : {odorless_mask.sum()}")
print(f"  purely odorless      : {purely_odorless}  <- dropped in A-drop, kept in A-keep")
print(f"  odorless + other fine: {mixed}             <- kept in both (drop only the odorless label)")

---
## Step 3 — Odorless strategy

### ▶ VARIANT A-KEEP
**Comment this entire block when building A-drop.**

`fine_odorless` is dropped and replaced by `meta_odorless` as a childless metacategory.  
All 4976 molecules are retained. The 199 purely-odorless molecules contribute only to the meta-level loss.

In [ ]:
# ── A-KEEP: comment out when building A-drop ──────────────────────────────

dataset = df.drop(columns=["fine_odorless"]).copy()
dataset["meta_odorless"] = odorless_mask.astype(int)

fine_final = [c for c in dataset.columns if c.startswith("fine_")]
meta_final = [c for c in dataset.columns if c.startswith("meta_")]
print(f"A-KEEP -> {len(dataset)} molecules | {len(fine_final)} fine | {len(meta_final)} meta")

dataset.to_csv("hmcn_dataset_A_keep.csv", index=False)
print("Saved -> hmcn_dataset_A_keep.csv")
# ── end A-KEEP ──────────────────────────────────────────────────────────────

### ▶ VARIANT A-DROP
**Comment this entire block when building A-keep.**

`fine_odorless` and the 199 purely-odorless molecules are removed.  
The 1 molecule with odorless + other fine labels is kept (only its odorless label is removed).

In [ ]:
# ── A-DROP: comment out when building A-keep ──────────────────────────────

drop_rows = df.index[(odorless_mask) & (other_sum == 0)]
dataset   = df.drop(index=drop_rows).drop(columns=["fine_odorless"]).reset_index(drop=True)

fine_final = [c for c in dataset.columns if c.startswith("fine_")]
meta_final = [c for c in dataset.columns if c.startswith("meta_")]
print(f"Dropped {len(drop_rows)} purely-odorless molecules")
print(f"A-DROP -> {len(dataset)} molecules | {len(fine_final)} fine | {len(meta_final)} meta")

dataset.to_csv("hmcn_dataset_A_drop.csv", index=False)
print("Saved -> hmcn_dataset_A_drop.csv")
# ── end A-DROP ──────────────────────────────────────────────────────────────

## Step 4 — Sanity checks
Run after whichever variant you built.

In [ ]:
fine_final = [c for c in dataset.columns if c.startswith("fine_")]
meta_final = [c for c in dataset.columns if c.startswith("meta_")]

# 1. All-zero fine rows should only be the meta-only molecules (not a bug)
zero_fine = (dataset[fine_final].sum(axis=1) == 0).sum()
print(f"Molecules with all-zero fine labels (meta-only): {zero_fine}")

# 2. Original meta columns are byte-for-byte unchanged
original_df   = pd.read_csv("hmcn_dataset.csv")
original_meta = [c for c in meta_cols if c in dataset.columns]
subset_orig   = original_df.set_index("SMILES").loc[dataset["SMILES"], original_meta].values
subset_new    = dataset.set_index("SMILES")[original_meta].values
assert (subset_orig == subset_new).all(), "Original meta columns changed!"
print("Original meta columns unchanged — OK")

# 3. Label prevalence summary
print(f"\n{'label':22s}{'positives':>10}{'%':>8}")
for c in sorted(meta_final) + sorted(fine_final):
    n = dataset[c].sum()
    print(f"{c:22s}{n:10d}{100*n/len(dataset):7.1f}%")